# External Evaluation — DeepSense 6G Scenario 9 (UNSEEN) — ResNet-50 vs. VIBE

Companion code for:

> **Look Once, Beam Twice: Camera-Primed Real-Time Double-Directional mmWave
> Beam Management for Vehicular Connectivity**
> Avhishek Biswas\*, Apala Pramanik\*, Eylem Ekici, Mehmet C. Vuran (\*equal contribution)
> *Proc. IEEE SECON 2026*, Pisa, Italy.
> Paper (arXiv): <https://arxiv.org/pdf/2605.05071>

**Scenario 9 is the unseen generalisation test**: neither the ResNet-50
baseline nor the VIBE models were trained on it. This notebook runs, on the
same images / ground-truth mmWave power vectors:

1. **ResNet-50** — vision baseline of Charan *et al.*, WCNC 2021 (ref. [22]).
2. **VIBE-YOLOR** — camera priming + radio projection only.
3. **VIBE-MA** — VIBE-YOLOR + closed-loop offset tracking.

Results feed the "Unseen" columns of Table II / Fig. 11.

### You must supply (not shipped in this repo)
- **DeepSense 6G Scenario 9** — https://www.deepsense6g.net/
- **ResNet-50 weights** (`CNN_beam_pred`) — upstream repo
  https://github.com/gourangc/Vision-Position-Beam-Prediction

Set the data location in the **CONFIG** cell (or `DEEPSENSE_SCENARIO9_ROOT`).

In [ ]:
!nvidia-smi

In [ ]:
# === Standard library ===
import os
import time
from glob import glob

# === Third-party ===
import numpy as np
import pandas as pd
from PIL import Image

# === PyTorch / vision ===
import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
from ultralytics import YOLO

In [ ]:
# =============================================================================
# CONFIG  --  edit paths / parameters in THIS cell only
# =============================================================================
# DeepSense 6G Scenario 9 (the UNSEEN generalisation scenario).
# Data you must supply yourself (NOT in this repo):
#   * DeepSense 6G Scenario 9         -> https://www.deepsense6g.net/
#   * ResNet-50 weights CNN_beam_pred -> https://github.com/gourangc/Vision-Position-Beam-Prediction
#
# Expected layout under scenario_root:
#   scenario9.csv
#   unit1/camera_data_passes/pass*/*.jpg
#   unit1/mmWave_data/*.txt
#   resources/annotations/bbox/*.txt
#   image_beam/saved_folder/best_model_ResNet50/checkpoint/CNN_beam_pred
# =============================================================================
import os
import sys
import pandas as pd

try:
    PROJECT_ROOT = os.path.dirname(os.path.abspath(__file__))
except NameError:
    PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Defaults to <project>/scenario9/scenario9_dev; override with:
#   export DEEPSENSE_SCENARIO9_ROOT=/data/DeepSense6G/scenario9/scenario9_dev
scenario_name = "Scenario9"
scenario_root = os.environ.get(
    "DEEPSENSE_SCENARIO9_ROOT",
    os.path.join(PROJECT_ROOT, "scenario9", "scenario9_dev"),
)

scenario_csv_path = os.path.join(scenario_root, "scenario9.csv")
base_dir          = os.path.join(scenario_root, "unit1", "camera_data_passes")
camera_data_root  = base_dir
rx_power_base     = os.path.join(scenario_root, "unit1", "mmWave_data")
bbox_base         = os.path.join(scenario_root, "resources", "annotations", "bbox")

model_path = os.path.join(
    scenario_root, "image_beam", "saved_folder",
    "best_model_ResNet50", "checkpoint", "CNN_beam_pred",
)

quantile_percentile = 0.80   # received-power quantile threshold: 0.80 / 0.90 / 0.95
delta               = 10     # VIBE-MA neighbour-search half-window (delta_max)

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "ResNet50ModelAnalysis")
os.makedirs(OUTPUT_DIR, exist_ok=True)

for _lbl,_p in [("scenario9.csv",scenario_csv_path),("camera_data_passes",base_dir),
                ("ResNet-50 CNN_beam_pred",model_path)]:
    if not os.path.exists(_p): print(f"[WARN] {_lbl} not found: {_p}")
print(f"[INFO] PROJECT_ROOT  = {PROJECT_ROOT}")
print(f"[INFO] scenario_root = {scenario_root}")
print(f"[INFO] OUTPUT_DIR    = {OUTPUT_DIR}")

scenario_df = pd.read_csv(scenario_csv_path)
scenario_df["image_name"] = scenario_df["unit1_rgb"].apply(os.path.basename)
image_to_power_path = dict(zip(scenario_df["image_name"], scenario_df["unit1_pwr_60ghz"]))

## 3. Helper functions — load bounding boxes & mmWave power

`load_bbox` reads the DeepSense bbox annotation; `load_mmwave_power` loads the
**64-element received-power vector** for an image (the ground truth: true beam
= its argmax, Q-threshold = a quantile of it). Paths come from CONFIG.

In [ ]:
# -----------------------------------------------------------------------------
# Helper readers (Section 3): load_bbox / load_mmwave_power. Paths from CONFIG.
# load_mmwave_power -> 64-element received-power vector (ground truth).
# -----------------------------------------------------------------------------
def load_bbox(image_name):
    ann_path = os.path.join(bbox_base, image_name.replace(".jpg", ".txt"))
    if not os.path.exists(ann_path):
        return None
    try:
        with open(ann_path, 'r') as f:
            line = f.readline().strip()
            vals = list(map(float, line.split()))
            if len(vals) == 5:
                _, x_c, y_c, w, h = vals
                x_min = x_c - w / 2
                x_max = x_c + w / 2
                return [x_min, y_c, x_max, y_c]
            elif len(vals) == 4:
                return vals
    except:
        return None
# === mmWave Power extraction helper function ===
def load_mmwave_power(image_name):
    try:
        rel_path = image_to_power_path.get(image_name, None)
        if rel_path is None or pd.isna(rel_path):
            return None
        rel_path = rel_path.lstrip("./")
        full_path = os.path.join(scenario_root, rel_path)
        if not os.path.exists(full_path):
            return None
        with open(full_path, 'r') as f:
            return np.array([float(line.strip()) for line in f if line.strip()])
    except:
        return None


## 4. ResNet-50 baseline — loader & top-k beam prediction

32-class **coarse** classifier (ref. [22]); class `k` → fine-beam pair
`(2k, 2k+1)` in the 64-beam codebook. `predict_beams_from_image` returns the
top-k coarse indices + confidences.

In [ ]:
# -----------------------------------------------------------------------------
# ResNet-50 vision baseline (Section 4) -- Charan et al., WCNC 2021, ref. [22].
# 32-class COARSE classifier: class k -> fine-beam pair (2k, 2k+1).
#
# build_net.py is a DeepSense 6G baseline artifact (the ResNet-50 builder by
# M. Alrabeiah) and is NOT redistributed in this repo. Download build_net.py
# from the upstream Vision-Position-Beam-Prediction repo and place it next to
# this notebook (the CONFIG cell already adds PROJECT_ROOT to sys.path):
#   https://github.com/gourangc/Vision-Position-Beam-Prediction
# -----------------------------------------------------------------------------
try:
    from build_net import resnet50  # ResNet-50 builder (DeepSense baseline)
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "build_net.py not found. It is a DeepSense 6G baseline artifact and is "
        "not shipped in this repo. Download build_net.py from "
        "https://github.com/gourangc/Vision-Position-Beam-Prediction and place "
        "it next to this notebook (the CONFIG cell adds PROJECT_ROOT to sys.path)."
    ) from exc

# Use GPU when available, else fall back to CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    print("[WARN] CUDA not available — ResNet-50 will run on CPU (slow).")
print(f"[INFO] Using device: {device}")

def load_model(model_path, num_classes=32):
    model = resnet50(pretrained=True, num_classes=num_classes)
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    print(f"[INFO] Model loaded. Model is on device: {next(model.parameters()).device}")
    return model

def preprocess_image(image_path):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    image = Image.open(image_path).convert('RGB')
    tensor = transform(image).unsqueeze(0).to(device)  # Add batch dim + move to device
    print(f"[INFO] Image tensor is on device: {tensor.device}")
    return tensor

def predict_beams_from_image(image_path, model_path, num_classes=32, topk=3):
    if not os.path.isfile(image_path):
        raise FileNotFoundError(f"Image file not found: {image_path}")
    
    model = load_model(model_path, num_classes=num_classes)
    
    try:
        image_tensor = preprocess_image(image_path)
    except Exception as e:
        raise RuntimeError(f"Failed to load image {image_path}: {e}")

    with torch.no_grad():
        _, logits = model(image_tensor)
        print(f"[INFO] Logits tensor is on device: {logits.device}")
        probs = F.softmax(logits, dim=1)
        topk_probs, topk_indices = torch.topk(probs, k=topk, dim=1)

    return {
        "image": os.path.basename(image_path),
        "topk_beams": topk_indices[0].tolist(),
        "confidences": [round(float(x), 6) for x in topk_probs[0].tolist()]
    }

## 5. VIBE-YOLOR — camera priming + radio-coordinate projection

`detect_car_bbox` (YOLOv11/COCO) + `estimate_beam_full_top3_from_box` (pinhole
projection onto the fixed `RX_BEAM_ANGLES` codebook, FOV=110°, W=960). Paper's
internal "VIBE-YOLOR" baseline (no closed loop).

In [ ]:
# -----------------------------------------------------------------------------
# VIBE-YOLOR (Section 5): camera priming + radio-coordinate projection, no loop.
# -----------------------------------------------------------------------------
# === OPTION 1: Load your custom-trained YOLOv8 model ===
# This model was trained by you on a custom dataset (e.g., 'radio', 'mmWave radio').
# custom_model_path = 'YOLOV8_trained_best.pt'
# yolo_model = YOLO(custom_model_path)
# print("Custom model classes:")
# print(yolo_model.model.names)

# === OPTION 2: Load a YOLOR model pretrained on COCO ===
from ultralytics import YOLO
import torch

# === Load native YOLOv11x pretrained model (on COCO dataset) ===
yolo_model = YOLO('yolo11x.pt')  # Large, accurate model trained on 80 COCO classes

# === Check if CUDA is available and print model device info ===
if torch.cuda.is_available():
    device_str = "cuda"
    print("🟢 CUDA is available. YOLOR will run on GPU.")
else:
    device_str = "cpu"
    print("⚠️ CUDA not available. YOLOR will run on CPU.")

# === Car Detection Function ===
def detect_car_bbox(image_path, car_class_id=2):
    """
    Detects the most confident car in the image and returns:
    [x_center_norm, y_center_norm, width_norm, height_norm, class_id]
    """
    results = yolo_model.predict(source=image_path, device=device_str, verbose=True)
    detections = results[0].boxes.data.cpu().numpy()  # [x1, y1, x2, y2, conf, class]

    image_width, image_height = results[0].orig_shape[1], results[0].orig_shape[0]
    best_conf = -1
    best_bbox = None

    for det in detections:
        x1, y1, x2, y2, conf, cls = det
        if int(cls) != car_class_id:
            continue

        if conf > best_conf:
            x_center = (x1 + x2) / 2 / image_width
            y_center = (y1 + y2) / 2 / image_height
            width = (x2 - x1) / image_width
            height = (y2 - y1) / image_height
            best_bbox = [x_center, y_center, width, height, int(cls)]
            best_conf = conf

    return best_bbox



# === Constants ===
FOV_DEG = 110.0
IMAGE_WIDTH = 960
RX_BEAM_ANGLES = [
    0.0, -45.0, -43.5, -42.1, -40.6, -39.2, -37.7, -36.3, -34.8, -33.4, -31.9,
    -30.5, -29.0, -27.6, -26.1, -24.7, -23.2, -21.8, -20.3, -18.9, -17.4,
    -16.0, -14.5, -13.1, -11.6, -10.2, -8.7, -7.3, -5.8, -4.4, -2.9, -1.5, 0.0,
     1.5, 2.9, 4.4, 5.8, 7.3, 8.7, 10.2, 11.6, 13.1, 14.5, 16.0, 17.4,
     18.9, 20.3, 21.8, 23.2, 24.7, 26.1, 27.6, 29.0, 30.5, 31.9, 33.4,
     34.8, 36.3, 37.7, 39.2, 40.6, 42.1, 43.5, 45.0
]


def estimate_beam_full_top3_from_box(image_name, box):
    """
    Estimate left/center/right beam index from YOLO box, and return full metadata.
    """
    power_vals = load_mmwave_power(image_name)
    if box is None or power_vals is None or len(power_vals) != 64:
        return {
            "image_name": image_name,
            "true_beam_index": None,
            "true_beam_power": None,
            "est_theta_deg_left": None,
            "est_theta_deg_center": None,
            "est_theta_deg_right": None,
            "est_beam_index_left": None,
            "est_beam_index_center": None,
            "est_beam_index_right": None,
            "est_beam_angle_left": None,
            "est_beam_angle_center": None,
            "est_beam_angle_right": None,
            "estimated_beam_power_left": None,
            "estimated_beam_power_center": None,
            "estimated_beam_power_right": None,
            "beam_index_error_left": None,
            "beam_index_error_center": None,
            "beam_index_error_right": None,
            "hit_top1": None,
            "hit_top2": None,
            "hit_top3": None
        }

    x_c, y_c, w, h, cls = box

    positions = {
        "left": x_c - w / 2,
        "center": x_c,
        "right": x_c + w / 2
    }

    est_data = {}

    for key, x_norm in positions.items():
        u = x_norm * IMAGE_WIDTH
        delta_u = u - (IMAGE_WIDTH / 2)
        theta_deg = (delta_u / IMAGE_WIDTH) * FOV_DEG

        closest_angle = min(RX_BEAM_ANGLES, key=lambda x: abs(x - theta_deg))
        beam_idx = RX_BEAM_ANGLES.index(closest_angle) + 1  # 1-based
        beam_error = None
        beam_power = None
        if power_vals is not None:
            beam_error = abs(beam_idx - int(np.argmax(power_vals) + 1))
            beam_power = round(power_vals[beam_idx - 1], 2)

        est_data[f"est_theta_deg_{key}"] = round(theta_deg, 2)
        est_data[f"est_beam_index_{key}"] = beam_idx
        est_data[f"est_beam_angle_{key}"] = round(closest_angle, 2)
        est_data[f"estimated_beam_power_{key}"] = beam_power
        est_data[f"beam_index_error_{key}"] = beam_error

    true_idx = int(np.argmax(power_vals) + 1)
    true_power = round(power_vals[true_idx - 1], 2)

    top3 = [
        est_data["est_beam_index_center"],
        est_data["est_beam_index_left"],
        est_data["est_beam_index_right"]
    ]
    top1 = top3[:1]
    top2 = top3[:2]

    return {
        "image_name": image_name,
        "true_beam_index": true_idx,
        "true_beam_power": true_power,
        **est_data,
        "hit_top1": int(true_idx in top1),
        "hit_top2": int(true_idx in top2),
        "hit_top3": int(true_idx in top3)
    }

## 6. VIBE-MA — closed-loop neighbour search + offset tracking

`correct_beam_with_offset_tracking_single_v2`: outward ±1,±2,…,±delta search
around `est_beam + current_offset`; first beam clearing the Q-threshold wins
and its offset is carried to the next frame in the same pass (Algorithm 1).

In [ ]:
# -----------------------------------------------------------------------------
# VIBE-MA closed loop (Section 6) -- Algorithm 1 in the paper.
# -----------------------------------------------------------------------------
def correct_beam_with_offset_tracking_single_v2(
    est_beam: int,
    true_beam: int,
    power_vec: np.ndarray,
    delta: int,
    pwr_threshold: float,
    current_offset: int = 0,
    clear_offset: bool = False
):
    """
    Beam correction using neighbor search and offset tracking.

    Args:
        est_beam: Estimated beam index (1-based).
        true_beam: Ground-truth beam index (1-based).
        power_vec: Beam power vector (64 values).
        delta: Maximum offset range to search (δmax).
        pwr_threshold: Minimum acceptable power.
        current_offset: Offset to apply to estimated beam.
        clear_offset: If True, resets offset to 0.

    Returns:
        corrected_beam_index, corrected_power, beams_searched, corrected_error, new_offset
    """

    # Step 0: Validate input
    if power_vec is None or len(power_vec) < 64:
        return None, None, None, None, 0

    # Step 1: Check estimated beam directly
    est_power = power_vec[est_beam - 1]
    if est_power >= pwr_threshold:
        corrected_error = abs(pwr_threshold - est_beam)
        return est_beam, est_power, 1, corrected_error, 0  # Use offset = 0 when no search needed

    # Step 2: Begin neighbor search around offset-adjusted center
    offset = 0 if clear_offset else current_offset
    center = est_beam + offset
    bmin, bmax = 1, len(power_vec)

    best_beam = center
    best_power = -float('inf')
    beams_checked = 0

    # Search ±delta around center
    for delta_i in range(1, delta + 1):
        for direction in [-1, 1]:
            b = center + direction * delta_i
            if not (bmin <= b <= bmax):
                continue

            beams_checked += 1
            power = power_vec[b - 1]

            # If threshold is met, return with offset
            if power >= pwr_threshold:
                corrected_error = abs(pwr_threshold - b)
                new_offset = b - est_beam  # update offset only if threshold met
                return b, power, beams_checked, corrected_error, new_offset

            # Track best power (for fallback only)
            if power > best_power:
                best_power = power
                best_beam = b

    # Step 3: Fallback – no beam met threshold, so keep offset = 0
    corrected_error = abs(true_beam - best_beam)
    return best_beam, best_power, beams_checked, corrected_error, 0

## 7. Run all passes — the evaluation loop

For every image of every `passNN` folder it runs ResNet-50, VIBE-YOLOR and
VIBE-MA on the same image / ground-truth power vector and records one CSV row.
Outage is cumulative (`topk_outage = top1<Q and … and topk<Q`). Output:
`ResNet50ModelAnalysis/Ext_Eval_Scenario9_VisionOnly_<Q>.csv`.

---

### What is a *pass*?

DeepSense 6G delivers each scenario as long, time-ordered capture sequences.
To evaluate **beam tracking under motion**, the frames are regrouped into
`passNN/` folders (built by `sort_data_in_pass.py` from `scenario9.csv` using
the dataset `seq_index`). One **pass = one continuous drive-by** of the
transmitter past the receiver: a contiguous, time-ordered run of frames
forming a single trajectory. The evaluation iterates pass-by-pass and the
**VIBE-MA offset state resets at the start of every pass** (each pass is an
independent trajectory — no offset should carry across passes).

In [ ]:
def process_image(idx, image_path, pass_name,
                  current_offset_resnet, current_offset_yolo):
    """
    Evaluate ResNet-50, VIBE-YOLOR and VIBE-MA on ONE image of
    Scenario 9 (the unseen scenario).

    Returns (row, current_offset_resnet, current_offset_yolo); row is the
    per-image result dict, or None if the image is skipped. The offsets
    are the VIBE-MA state carried to the next image in the same pass.
    """
    image_name = os.path.basename(image_path)

    print("------------------------------------------------------------------------------------------------")
    print("------------------------------------------------------------------------------------------------")
    
    try:
        mmwave_path = os.path.join(scenario_root, image_to_power_path[image_name].lstrip("./"))
        power_vec = load_mmwave_power(image_name)
    except Exception as e:
        print(f"⚠️ {image_name} → Cannot load mmWave power: {e}")
        return None, current_offset_resnet, current_offset_yolo

    # Validate power vector
    if power_vec is None or len(power_vec) != 64:
        print(f"❌ Skipping {image_name}: Invalid or missing power vector.")
        return None, current_offset_resnet, current_offset_yolo

    # === Compute true beam and threshold ===
    true_beam = int(np.argmax(power_vec) + 1)
    true_power = round(power_vec[true_beam - 1], 3)
    adaptive_threshold = round(np.quantile(power_vec, quantile_percentile), 4)
    max_power_all_beams = np.max(power_vec)

    print("\n mmWave Vector Stats:")
    print(f"   - Max Beam Power: {max_power_all_beams:.3f}")
    print(f"   - True Beam: {true_beam}")
    print(f"   - True Beam Power: {true_power:.3f}")
    print(f"   - Quantile Threshold ({int(quantile_percentile*100)}%): {adaptive_threshold:.4f}")

    if true_power < adaptive_threshold:
        print(f"⚠️ Skipping {image_name}: True beam power ({true_power:.6f}) < Quantile threshold ({adaptive_threshold:.6f})")
        return None, current_offset_resnet, current_offset_yolo

    # === Initialize Variables ===
    resnet_top1 = resnet_top2 = resnet_top3 = None
    resnet_beam_index_error = 0
    resnet_beam_power_error = 0.0
    resnet_time = 0.0
    resnet_corr_time = 0.0
    corrected_resnet = (0, 0.0, 0, 0, 0)
    resnet_corr_power_diff = 0.0
    
    # --- YOLOR ---
    center = left = right = 0
    yolo_beam_index_error = 0
    yolo_beam_power_error = 0.0
    detect_time = 0.0
    yolo_beam_time = 0.0
    yolo_corr_time = 0.0
    corrected_yolo = (0, 0.0, 0, 0, 0)
    yolo_corr_power_diff = 0.0

    # ==========================================================
     # === PIPELINE: RESNET50 → Vision Model Prediction ===
    # ==========================================================
    try:
        print("\n [RESNET] Starting vision-based beam prediction...")
        t_resnet_start = time.time()
        
        top_k_resnet = predict_beams_from_image(image_path, model_path)
        resnet_time = round(time.time() - t_resnet_start, 4)
    
        resnet_top1 = top_k_resnet["topk_beams"][0]
        resnet_top2 = top_k_resnet["topk_beams"][1]
        resnet_top3 = top_k_resnet["topk_beams"][2]

        # === Power from top-k predicted coarse indices ===
        resnet_power_top1 = power_vec[2 * resnet_top1] if resnet_top1 is not None else 0
        resnet_power_top2 = power_vec[2 * resnet_top2] if resnet_top2 is not None else 0
        resnet_power_top3 = power_vec[2 * resnet_top3] if resnet_top3 is not None else 0

        # === Outage logic ===
        resnet_top1_outage = resnet_power_top1 < adaptive_threshold
        resnet_top2_outage = resnet_top1_outage and (resnet_power_top2 < adaptive_threshold)
        resnet_top3_outage = resnet_top2_outage and (resnet_power_top3 < adaptive_threshold)

        # === Pick the better of two adjacent fine beams for top1 ===
        beam_range = [2 * resnet_top1, 2 * resnet_top1 + 1]
        Resnetpredicted_power = max(power_vec[i] for i in beam_range)
        best_resnet_beam = max(beam_range, key=lambda i: power_vec[i])
        resnet_beam_index_error = abs(true_beam - best_resnet_beam)
        resnet_beam_power_error = round(abs(true_power - Resnetpredicted_power), 2)
    
        print(f" [RESNET] Top-3 predicted coarse beam indices: {resnet_top1}, {resnet_top2}, {resnet_top3}")
        print(f" [RESNET] Top-1 Power: {resnet_power_top1:.3f} | Top-2: {resnet_power_top2:.3f} | Top-3: {resnet_power_top3:.3f}")
        print(f" [RESNET] Outage Flags: Top-1: {resnet_top1_outage}, Top-2: {resnet_top2_outage}, Top-3: {resnet_top3_outage}")
        print(f" [RESNET] Best fine beam: {best_resnet_beam} | Power: {Resnetpredicted_power:.3f}")
        print(f" [RESNET] Beam index error: {resnet_beam_index_error} | Power error: {resnet_beam_power_error}")
        print(f" [RESNET] Prediction time: {resnet_time} seconds")
    
    except Exception as e:
        resnet_top1 = resnet_top2 = resnet_top3 = None
        resnet_power_top1 = resnet_power_top2 = resnet_power_top3 = 0
        resnet_top1_outage = resnet_top2_outage = resnet_top3_outage = False
        resnet_beam_index_error = 0
        resnet_beam_power_error = 0
        resnet_time = 0.0
        print(f"❌ [RESNET] Pipeline failed: {e}")

    # ==========================================================
     # === PIPELINE: VIBE-YOLOR → Vision-Based Beamforming ===
    # ==========================================================
    try:
        print("------------------------------------------------------------------------------------------------")
        t_yoloest = time.time()
        bbox = detect_car_bbox(image_path)
        detect_time = round(time.time() - t_yoloest, 4)
        print(f"📦 BBox detection time: {detect_time} sec")
        print(f"📦 Detected BBox: {bbox}")
    
        if bbox is None:
            print(f"❌ No car detected in {image_name}")
            return None, current_offset_resnet, current_offset_yolo
    
        t_beamest = time.time()
        yolor_result = estimate_beam_full_top3_from_box(image_name, bbox)
        yolo_beam_time = round(time.time() - t_beamest, 4)
    
        # Extract beam indices
        left = yolor_result["est_beam_index_left"]
        center = yolor_result["est_beam_index_center"]
        right = yolor_result["est_beam_index_right"]
    
        # Compute powers for each top K 
        yolo_power_left = power_vec[left - 1] if 1 <= left <= 64 else 0
        yolo_power_center = power_vec[center - 1] if 1 <= center <= 64 else 0
        yolo_power_right = power_vec[right - 1] if 1 <= right <= 64 else 0
    
        # Compute outage flags
        yolo_top1_outage = yolo_power_center < adaptive_threshold
        yolo_top2_outage = yolo_top1_outage and (yolo_power_left < adaptive_threshold)
        yolo_top3_outage = yolo_top2_outage and (yolo_power_right < adaptive_threshold)

        print(f"\n YOLOR Predicted Beams:")
        print(f"   - Left: {left} (Power = {yolo_power_left:.3f}, Outage = {yolo_top2_outage})")
        print(f"   - Center: {center} (Power = {yolo_power_center:.3f}, Outage = {yolo_top1_outage})")
        print(f"   - Right: {right} (Power = {yolo_power_right:.3f}, Outage = {yolo_top3_outage})")
        
        print(f" Beam estimation time: {yolo_beam_time} sec")
        
    except Exception as e:
        print(f"❌ YOLOR beam estimation failed: {e}")
        return None, current_offset_resnet, current_offset_yolo


    # ==========================================================
     # === VIBE-MA Correction ===
    # ==========================================================
    try:
        # print("------------------------------------------------------------------------------------------------")
        est_power = power_vec[center - 1] if 1 <= center <= 64 else 0
        max_power = np.max(power_vec)

        print(f"\n [YOLOR CORRECTION] Correction Logic")
        print(f"   - est_power (center): {est_power:.4f}")
        print(f"   - max_power: {max_power:.4f}")
        print(f"   - threshold: {adaptive_threshold:.4f}")
        print(f"   - current_offset: {current_offset_yolo}")

        if max_power < adaptive_threshold or est_power >= adaptive_threshold:
            corrected_yolo = (center, est_power, 0, abs(true_beam - center), 0)
            yolo_corr_time = 0.0
            reason = "max SNR too low" if max_power < adaptive_threshold else "center beam is already sufficient"
            print(f"⚠️ [YOLOR CORRECTION] No correction needed — {reason}")
        else:
            t_yolo_corr = time.time()
            corrected_yolo = correct_beam_with_offset_tracking_single_v2(
                est_beam=center,
                true_beam=true_beam,
                power_vec=power_vec,
                delta=delta,
                pwr_threshold=adaptive_threshold,
                current_offset=current_offset_yolo,
                clear_offset=(idx == 0)
            )
            yolo_corr_time = round(time.time() - t_yolo_corr, 6)
            new_beam = corrected_yolo[0]
            new_offset = new_beam - center
            current_offset_yolo = new_offset

            print(f" [YOLOR CORRECTION] Correction Applied:")
            print(f"   - Corrected Beam: {new_beam}")
            print(f"   - Corrected Power: {corrected_yolo[1]:.6f}")
            print(f"   - Correction Error (true vs corrected): {corrected_yolo[3]}")
            print(f"   - Beams Searched: {corrected_yolo[2]}")
            print(f"   - New Offset: {new_offset}")
            print(f"   - Correction Time: {yolo_corr_time:.4f} sec")

        yolo_corr_power_diff = round(adaptive_threshold - corrected_yolo[1], 6)
        yolor_corr_outage = corrected_yolo[1] < adaptive_threshold

        print(f"[YOLOR CORRECTION] Power Gap (adaptive - corrected): {yolo_corr_power_diff:.6f}")
        print(f"{' Outage after correction' if yolor_corr_outage else '✅ Correction successful — power sufficient'}")

    except Exception as e:
        corrected_yolo = (center, est_power, 0, abs(true_beam - center), 0)
        yolo_corr_time = 0.0
        yolo_corr_power_diff = round(adaptive_threshold - est_power, 6)
        yolor_corr_outage = True
        print(f"\n❌ [YOLOR CORRECTION] Failed: {e}")
        print(" Treating correction failure as outage.")


    # ==========================================================
     # === Correction: ResNet + M.Avg ===
    # ==========================================================
    try:
        print("\n [RESNET CORRECTION] Starting correction using M.Avg...")
    
        beam_range = [2 * resnet_top1, 2 * resnet_top1 + 1]
        resnet_est = max(beam_range, key=lambda i: power_vec[i])
        est_power = power_vec[resnet_est]
        max_power = np.max(power_vec)

        if max_power < adaptive_threshold or est_power >= adaptive_threshold:
            corrected_resnet = (resnet_est, est_power, 0, abs(true_beam - resnet_est), 0)
            resnet_corr_time = 0.0
            print("⚠️ [RESNET CORRECTION] No correction needed (already above threshold or poor max power).")
        else:
            t4 = time.time()
            corrected_resnet = correct_beam_with_offset_tracking_single_v2(
                est_beam=resnet_est,
                true_beam=true_beam,
                power_vec=power_vec,
                delta=delta,
                pwr_threshold=adaptive_threshold,
                current_offset=current_offset_resnet,
                clear_offset=(idx == 0)
            )
            resnet_corr_time = round(time.time() - t4, 6)
            current_offset_resnet = corrected_resnet[0] - resnet_est

            print(f" [RESNET CORRECTION] Corrected beam: {corrected_resnet[0]} | Offset applied: {current_offset_resnet}")
            print(f" [RESNET CORRECTION] Power: {corrected_resnet[1]:.3f} | Beams searched: {corrected_resnet[2]} | Error index: {corrected_resnet[3]}")
            print(f" [RESNET CORRECTION] Correction time: {resnet_corr_time}s")

        resnet_corr_power_diff = round(adaptive_threshold - corrected_resnet[1], 6)
        resnet_corr_outage = corrected_resnet[1] < adaptive_threshold

        print(f"[RESNET CORRECTION] Power Difference (adaptive - corrected): {resnet_corr_power_diff:.6f}")
        print(f"{' Outage after correction' if resnet_corr_outage else '✅ Correction successful — power sufficient'}")

    except Exception as e:
        corrected_resnet = (resnet_est, est_power, 0, abs(true_beam - resnet_est), 0)
        resnet_corr_time = 0.0
        resnet_corr_power_diff = round(adaptive_threshold - est_power, 6)
        resnet_corr_outage = True
        print(f"\n❌ [RESNET CORRECTION] Failed: {e}")
        print(" Treating failed correction as outage.")

    
    # === Log this image result ===
    row = {
        # Data
        "image_name": image_name,
        "pass_name": pass_name,
        "mmWave_Data": power_vec,
        "true_beam_index": true_beam,
        "true_beam_power": true_power,
        "QuantileThreshold": adaptive_threshold,

        # Vision Model (ResNet)
        "resnet_top1": resnet_top1,
        "resnet_top2": resnet_top2,
        "resnet_top3": resnet_top3,
        "resnet_top1_power": round(resnet_power_top1, 6),
        "resnet_top2_power": round(resnet_power_top2, 6),
        "resnet_top3_power": round(resnet_power_top3, 6),
        "resnet_top1_outage": resnet_top1_outage,
        "resnet_top2_outage": resnet_top2_outage,
        "resnet_top3_outage": resnet_top3_outage,
        "resnet_beam_index_error": resnet_beam_index_error,
        "resnet_beam_power_error": round(resnet_beam_power_error, 3),
        "resnet_corr_beam": corrected_resnet[0],
        "resnet_corr_error_index": corrected_resnet[3],
        "resnet_corr_beams_searched": corrected_resnet[2],
        "resnet_corr_power": round(corrected_resnet[1], 3),
        "resnet_corr_power_diff": resnet_corr_power_diff,
        "resnet_corr_offset_applied": current_offset_resnet,
        "resnet_corr_outage": resnet_corr_outage,

        # VIBE Model
        "yolor_center": center,
        "yolor_top1_power": round(yolo_power_center, 6),
        "yolor_top1_outage": yolo_top1_outage,

        # VIBE-MA Model
        "yolor_beam_index_error": yolo_beam_index_error,
        "yolor_beam_power_error": round(yolo_beam_power_error, 3),
        "yolor_corr_beam": corrected_yolo[0],
        "yolor_corr_error_index": corrected_yolo[3],
        "yolor_corr_beams_searched": corrected_yolo[2],
        "yolor_corr_power": round(corrected_yolo[1], 3),
        "yolor_corr_power_diff": yolo_corr_power_diff,
        "yolor_corr_offset_applied": current_offset_yolo,
        "yolor_corr_outage": yolor_corr_outage,

        # Timing
        "timing_resnet_pred": resnet_time,
        "timing_resnet_correction": resnet_corr_time,
        "timing_yolo_detect": detect_time,
        "timing_yolo_beam": yolo_beam_time,
        "timing_yolo_correction": yolo_corr_time,
    }

    return row, current_offset_resnet, current_offset_yolo

In [ ]:
# =============================================================================
# Run the evaluation over every pass / image. process_image() (cell above)
# does all per-image work; here we enumerate passes, reset the VIBE-MA offsets
# per pass, thread them image-to-image, collect rows, write the per-Q CSV.
# =============================================================================
all_results = []

all_pass_folders = sorted([
    f for f in os.listdir(base_dir)
    if os.path.isdir(os.path.join(base_dir, f)) and f.startswith("pass")
])

for pass_name in all_pass_folders:
    pass_path = os.path.join(camera_data_root, pass_name)
    image_paths = sorted(glob(os.path.join(pass_path, "*.jpg")))
    if not image_paths:
        print(f"[WARN] No images found in {pass_name}")
        continue
    print(f"\n[INFO] {pass_name}: {len(image_paths)} images")

    current_offset_resnet = 0
    current_offset_yolo = 0

    for idx, image_path in enumerate(image_paths):
        row, current_offset_resnet, current_offset_yolo = process_image(
            idx, image_path, pass_name,
            current_offset_resnet, current_offset_yolo,
        )
        if row is not None:
            all_results.append(row)

df_all = pd.DataFrame(all_results)
csv_path = os.path.join(
    OUTPUT_DIR, f"Ext_Eval_Scenario9_VisionOnly_{quantile_percentile}.csv"
)
df_all.to_csv(csv_path, index=False)
print(f"\n[INFO] All results saved to: {csv_path}")


# Calculate Outage Probability and Print Success Rate

In [ ]:
# Roll up the per-Q result CSVs into one outage / timing summary
# (feeds the "Unseen" columns of Table II / Fig. 11 of the paper).

files = {
    "80": os.path.join(OUTPUT_DIR, "Ext_Eval_Scenario9_VisionOnly_0.80.csv"),
    "90": os.path.join(OUTPUT_DIR, "Ext_Eval_Scenario9_VisionOnly_0.90.csv"),
    "95": os.path.join(OUTPUT_DIR, "Ext_Eval_Scenario9_VisionOnly_0.95.csv"),
}
outage_metrics = {
    "yolor_top1_outage": "YOLO_Top1",
    "yolor_corr_outage": "YOLO_Corr",
    "resnet_top1_outage": "ResNet_Top1",
    "resnet_top2_outage": "ResNet_Top2",
    "resnet_top3_outage": "ResNet_Top3",
    "resnet_corr_outage": "ResNet_Corr",
}
timing_metrics = {
    "YOLO TopKTime":   ["timing_yolo_detect", "timing_yolo_beam"],
    "YOLO CorrTime":   ["timing_yolo_detect", "timing_yolo_beam", "timing_yolo_correction"],
    "ResNet TopKTime": ["timing_resnet_pred"],
    "ResNet CorrTime": ["timing_resnet_pred", "timing_resnet_correction"],
}

summary = []
for snr_value, path in files.items():
    df = pd.read_csv(path)
    row = {}
    for col, label in outage_metrics.items():
        row[label] = round(100 * df[col].sum() / len(df), 2) if col in df.columns else "N/A"
    for method, cols in timing_metrics.items():
        row[method] = round(df[cols].sum(axis=1).mean(), 4) if all(c in df.columns for c in cols) else "N/A"
    summary.append(pd.Series(row, name=int(snr_value)))

final_df = pd.DataFrame(summary)
summary_path = os.path.join(OUTPUT_DIR, "combined_outage_timing_summary_Scenario9.csv")
final_df.to_csv(summary_path)
print(f"[INFO] Wrote summary -> {summary_path}")
print(final_df)